# Data Science Programming - Sesion 5 (via avanzada)

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 22 de septiembre, 2026**

---

## Antes de empezar, lee esto

Este notebook **reemplaza** al de la sesion. Si ya programas, trabaja aqui a tu ritmo
mientras el resto del grupo ve los fundamentos de matplotlib.

- **No vale nota.** Existe para que no te toque esperar.
- **Trabajas solo.** Durante la clase estare con quienes estan empezando, asi que este
  notebook trae todo lo que necesitas. Cada reto se verifica solo: corres la celda de
  verificacion y sabes si lo lograste.
- **No tienes que entregarlo.** Estos retos no se califican. Subelo al aula solo si quieres que lo revise y te comente por escrito: es totalmente opcional y no hay que exponer.
- Si un reto se te atraviesa, sigue con el siguiente y vuelve despues.

Cubre el mismo temario que la clase normal, asi que no te quedas por fuera de nada que
necesites para la **fase de visualizacion del Proyecto (Actividad 3)**. Solo que llegas mas lejos: la clase normal dibuja
graficos; aqui vas a escribir funciones que **construyen** graficos con un contrato exacto,
que es lo que se hace cuando el grafico tiene que salir igual la semana entrante.

---
## La regla que cambia todo en esta sesion

En la clase normal el codigo termina en `plt.show()`. Aqui **no**.

```python
# lo que hace la clase normal
def mi_grafico(datos):
    plt.bar(datos.keys(), datos.values())
    plt.show()          # y aqui se acabo: no queda nada que tocar

# lo que vas a hacer tu
def mi_grafico(datos):
    fig, ax = plt.subplots()
    ax.bar(datos.keys(), datos.values())
    return fig, ax      # el objeto sobrevive: se puede inspeccionar, reusar y testear
```

Dos razones, una practica y una que te va a servir hoy mismo:

1. Quien recibe `fig, ax` puede seguir trabajando: cambiarle el titulo, meterlo en un panel,
   guardarlo con `fig.savefig(...)`, aplicarle un tema. Con `plt.show()` el grafico ya se fue.
2. **Un grafico se puede verificar con `assert`.** No comparando imagenes pixel a pixel:
   eso es fragil y no dice nada. Se verifica el **objeto** `Axes`, que expone todo lo que
   le dibujaste encima.

Toda la verificacion de este notebook se apoya en esa segunda idea.

---
## Repaso relampago

Lo que la clase normal ve en la primera hora, en tablas. Si algo de esto no te suena, mejor
toma el notebook normal.

### Anatomia

| Pieza | Que es |
|---|---|
| `Figure` | el lienzo completo. Tiene tamano en pulgadas y DPI |
| `Axes` | **una** region con datos dibujados. Es lo que uno llama "el grafico" |
| `Axis` | el eje X o el eje Y de un `Axes`: escala, limites, ticks, formateador |
| `fig, ax = plt.subplots()` | la unica forma de arrancar que deberias usar |
| `plt.subplots(2, 2)` | devuelve `(fig, array_de_4_axes)` |
| Interfaz `pyplot` vs `Axes` | `plt.title(...)` actua sobre el eje "actual"; `ax.set_title(...)` sobre uno explicito. Con mas de un eje, la primera es una trampa |

### Que grafico para que pregunta

| Pregunta | Grafico | Metodo |
|---|---|---|
| Como se comparan unas categorias | barras | `ax.bar` / `ax.barh` |
| Como evoluciona algo en el tiempo | linea | `ax.plot` |
| Como se distribuye una variable | histograma | `ax.hist` |
| Se relacionan dos variables | dispersion | `ax.scatter` |
| Como se distribuye por grupo | caja | `ax.boxplot` / `sns.boxplot` |
| Como se relacionan muchas variables | mapa de calor | `sns.heatmap(df.corr())` |

Torta: casi nunca. El ojo humano compara longitudes bien y angulos mal.

### Personalizacion

| Que | Como |
|---|---|
| Textos | `ax.set_title`, `ax.set_xlabel`, `ax.set_ylabel` |
| Leyenda | pasar `label=` al dibujar y despues `ax.legend()` |
| Limites | `ax.set_ylim(0, 100)` |
| Ticks | `ax.set_xticks(pos)` + `ax.set_xticklabels(txt, rotation=45, ha='right')` |
| Formato de ticks | `ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:,.0f}'))` |
| Etiqueta sobre la barra | `ax.bar_label(barras, labels=[...], padding=3)` |
| Marco | `ax.spines['top'].set_visible(False)` |
| Rejilla | `ax.grid(axis='y', alpha=0.3)` + `ax.set_axisbelow(True)` |
| Segundo eje Y | `ax2 = ax.twinx()` |
| Titulo de la figura | `fig.suptitle('...')` |
| Que no se pisen | `fig.tight_layout()` |

### Seaborn

| Pieza | Nota |
|---|---|
| `sns.barplot(data=df, x=..., y=..., ax=ax)` | agrega y ademas dibuja el intervalo de confianza |
| `sns.histplot`, `sns.boxplot`, `sns.scatterplot` | reciben `ax=` y **devuelven ese mismo `Axes`** |
| `sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')` | el mapa de correlacion |
| `sns.pairplot`, `sns.lmplot`, `sns.catplot` | son de nivel figura: crean su propia `Figure`, **no** aceptan `ax=` |
| `sns.set_theme(style='whitegrid')` | cambia el estilo global del proceso |

La distincion nivel-eje contra nivel-figura es la que mas confunde: si una funcion de seaborn
no acepta `ax=`, es porque construye la figura entera y te devuelve un `FacetGrid` o similar,
no un `Axes`.

### Y lo que la clase normal no ve: como se interroga un `Axes`

Esta tabla es la herramienta de trabajo del notebook. Todo lo que dibujas queda guardado en
el objeto y se puede leer.

| Pregunta | Como se lee |
|---|---|
| Que titulo tiene | `ax.get_title()`, `ax.get_xlabel()`, `ax.get_ylabel()` |
| Cuantas lineas dibuje | `len(ax.get_lines())` |
| Con que datos | `ax.get_lines()[0].get_ydata()`, `.get_xdata()`, `.get_marker()` |
| Cuantas barras y de que altura | `len(ax.patches)`, `[p.get_height() for p in ax.patches]` |
| Cuantos scatter y con que puntos | `len(ax.collections)`, `ax.collections[0].get_offsets()` |
| Que textos sueltos hay | `[t.get_text() for t in ax.texts]` (ahi caen `ax.text` y `ax.bar_label`) |
| Hay leyenda y que dice | `ax.get_legend()`, `[t.get_text() for t in ax.get_legend().get_texts()]` |
| Como quedaron los limites | `ax.get_ylim()`, `ax.get_xlim()` |
| Que dicen los ticks | `[t.get_text() for t in ax.get_xticklabels()]` |
| Con que formato | `f = ax.yaxis.get_major_formatter()` y despues `f(1500000, None)` |
| Se ve el marco de arriba | `ax.spines['top'].get_visible()` |
| Hay rejilla horizontal | `ax.get_ygridlines()[0].get_visible()` |
| Cuantos ejes tiene la figura | `len(fig.axes)` |
| De que tamano es | `fig.get_size_inches()` |
| Que titulo general | `[t.get_text() for t in fig.texts]` |

**Un detalle que muerde:** los textos de los ticks pueden venir vacios hasta que la figura se
dibuja. Antes de leer `get_xticklabels()`, llama `fig.canvas.draw()`. En las celdas de
verificacion ya esta puesto.

---
## Los datos

Cuatro retos sobre el mismo caso: una cadena de tiendas en Boyaca, primer semestre de 2026.
Los ingresos y las devoluciones estan escritos a mano, no bajados de internet, por la misma
razon de siempre: **sabemos exactamente cuanto valen**, asi que los asserts pueden comprobar
que el grafico dibujo el numero correcto y no uno parecido.

Las unidades, la satisfaccion y las devoluciones mensuales se derivan con una semilla fija,
asi que a todos nos da igual.

Ejecuta esta celda antes de empezar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Ingresos mensuales en pesos. El orden del diccionario NO es el orden de la respuesta:
# varios retos exigen ordenar. Si te apoyas en este orden, el assert te lo cobra.
INGRESOS = {
    'Sogamoso':       [6100000, 6450000, 5980000, 6700000, 7050000, 6820000],
    'Villa de Leyva': [2100000, 2450000, 2680000, 3100000, 4050000, 3860000],
    'Tunja':          [12400000, 13100000, 11850000, 14200000, 15050000, 13750000],
    'Chiquinquira':   [3200000, 3050000, 3480000, 3610000, 3390000, 3720000],
    'Duitama':        [8300000, 7950000, 8600000, 9100000, 8850000, 9400000],
}
MESES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun']

# Total del semestre por ciudad: {ciudad: pesos}
TOTAL_CIUDAD = {c: sum(v) for c, v in INGRESOS.items()}

# Causas de devolucion del semestre, contadas en el sistema de la cadena.
DEVOLUCIONES = {
    'Empaque danado':        34,
    'Producto vencido':      121,
    'Cantidad incorrecta':   58,
    'Referencia equivocada': 87,
    'Precio mal marcado':    19,
    'Entrega tardia':        11,
    'Otro':                  6,
}

# Tabla larga para el reto abierto y para el mapa de correlacion.
TICKET = {'Tunja': 62000, 'Duitama': 55000, 'Sogamoso': 48000,
          'Chiquinquira': 41000, 'Villa de Leyva': 78000}
_rng = np.random.default_rng(2026)
_filas = []
for _ciudad, _serie in INGRESOS.items():
    for _i, _mes in enumerate(MESES):
        _ing = _serie[_i]
        _uni = int(round(_ing / TICKET[_ciudad] * _rng.uniform(0.88, 1.12)))
        _dev = int(round(_uni * _rng.uniform(0.01, 0.06)))
        _filas.append({
            'ciudad': _ciudad, 'mes': _mes, 'mes_num': _i + 1,
            'ingreso': _ing, 'unidades': _uni, 'devoluciones': _dev,
            'satisfaccion': round(float(np.clip(4.6 - _dev / max(_uni, 1) * 12
                                                + _rng.normal(0, 0.18), 1, 5)), 2),
        })
ventas = pd.DataFrame(_filas)

print(ventas.shape)
print(ventas.head())
print()
for c, t in sorted(TOTAL_CIUDAD.items(), key=lambda kv: -kv[1]):
    print(f'{c:16} {t:>12,}')

---
## Interludio - un grafico interrogado

Antes del primer reto, mira la tecnica funcionando. Esta celda dibuja el mapa de correlacion
que la clase normal ve al final, y despues **lo interroga**. No hay nada que resolver aqui:
es para que veas que un grafico de seaborn tambien es un `Axes` normal y corriente.

Ojo con `cbar=False`: la barra de color de un `heatmap` es **otro `Axes`** dentro de la misma
figura. Si la dejas, `len(fig.axes)` da 2 y no 1. Ese tipo de sorpresa es la razon por la que
conviene mirar el objeto en vez de la imagen.

In [ ]:
numericas = ['ingreso', 'unidades', 'devoluciones', 'satisfaccion']
corr = ventas[numericas].corr()

fig_demo, ax_demo = plt.subplots(figsize=(5.5, 4.5))
resultado = sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                        vmin=-1, vmax=1, cbar=False, ax=ax_demo)
ax_demo.set_title('Correlacion entre variables')

print('sns.heatmap devolvio el mismo Axes que le pase:', resultado is ax_demo)
print('ejes en la figura:', len(fig_demo.axes), '(seria 2 si dejara cbar=True)')
print('mallas de color dibujadas:', len(ax_demo.collections))
print('forma de la matriz pintada:', ax_demo.collections[0].get_array().shape)
print('numeros anotados encima:', len(ax_demo.texts))
print('etiquetas del eje X:', [t.get_text() for t in ax_demo.get_xticklabels()])
print('titulo:', repr(ax_demo.get_title()))
plt.show()

---
## Reto 1 - Barras con contrato milimetrico

Escribe `grafico_barras(totales)`.

Es un grafico de barras corriente, salvo que **el contrato es exacto**. No es capricho: asi
se piden los graficos cuando alguien mas los va a usar en un informe.

**Contrato:**
- Recibe un diccionario `{ciudad: ingreso_total}` (usa `TOTAL_CIUDAD`).
- Devuelve la tupla `(fig, ax)`. No llames `plt.show()` dentro de la funcion.

| Requisito | Valor exacto |
|---|---|
| Tamano de la figura | `(9, 5)` pulgadas |
| Numero de barras | una por ciudad |
| Orden | por ingreso **descendente** de izquierda a derecha |
| Titulo | `'Ingresos por ciudad - primer semestre 2026'` |
| Etiqueta X | `'Ciudad'` |
| Etiqueta Y | `'Ingresos (COP)'` |
| Etiquetas de tick X | los nombres de ciudad, en el orden de las barras |
| Texto sobre cada barra | el valor con separador de miles: `'80,350,000'` |
| Leyenda | visible, con una unica entrada que diga `'Ingresos'` |
| Formato del eje Y | separador de miles y sin decimales |
| Limite superior del eje Y | al menos `1.10 * el valor mas alto`, para que el texto no se salga |

Dos cosas que probablemente tengas que buscar:

- El texto encima de la barra sale con `ax.bar_label(barras, labels=[...], padding=3)`, donde
  `barras` es lo que devuelve `ax.bar(...)`. Tambien vale hacerlo a mano con `ax.text`.
- El formato del eje se cambia con
  `ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: ...))`. El assert va a
  llamar a ese formateador con `f(1500000, None)` y espera `'1,500,000'`; cualquier
  formateador que responda eso sirve.

In [ ]:
def grafico_barras(totales):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 1 ---
fig1, ax1 = grafico_barras(TOTAL_CIUDAD)
fig1.canvas.draw()   # sin esto los textos de los ticks pueden venir vacios

esperado = sorted(TOTAL_CIUDAD.items(), key=lambda kv: -kv[1])
nombres_esp = [c for c, _ in esperado]
valores_esp = [v for _, v in esperado]

assert isinstance(fig1, matplotlib.figure.Figure), 'El primer valor devuelto debe ser la Figure'
assert isinstance(ax1, matplotlib.axes.Axes), 'El segundo valor devuelto debe ser el Axes'
assert tuple(round(x, 3) for x in fig1.get_size_inches()) == (9.0, 5.0), \
    f'La figura debe medir 9x5 pulgadas, no {tuple(fig1.get_size_inches())}'

assert len(ax1.patches) == len(TOTAL_CIUDAD), \
    f'Deben quedar {len(TOTAL_CIUDAD)} barras en ax.patches, hay {len(ax1.patches)}'
alturas = [round(p.get_height()) for p in ax1.patches]
assert alturas == valores_esp, \
    f'Las barras deben ir de mayor a menor ingreso.\n  esperado: {valores_esp}\n  obtenido: {alturas}'

etiquetas_x = [t.get_text() for t in ax1.get_xticklabels()]
assert etiquetas_x == nombres_esp, \
    f'Los ticks del eje X deben ser las ciudades en el orden de las barras.\n  esperado: {nombres_esp}\n  obtenido: {etiquetas_x}'

assert ax1.get_title() == 'Ingresos por ciudad - primer semestre 2026', \
    f'Titulo exacto del contrato. Obtenido: {ax1.get_title()!r}'
assert ax1.get_xlabel() == 'Ciudad', f'Etiqueta X esperada Ciudad, obtenida {ax1.get_xlabel()!r}'
assert ax1.get_ylabel() == 'Ingresos (COP)', \
    f'Etiqueta Y esperada Ingresos (COP), obtenida {ax1.get_ylabel()!r}'

textos = sorted(t.get_text() for t in ax1.texts)
assert textos == sorted(f'{v:,}' for v in valores_esp), \
    ('Falta el valor encima de cada barra con separador de miles (ej. 80,350,000). '
     f'En ax.texts hay: {textos}')

leyenda = ax1.get_legend()
assert leyenda is not None, 'Falta la leyenda: pasa label= al dibujar y despues llama ax.legend()'
entradas = [t.get_text() for t in leyenda.get_texts()]
assert entradas == ['Ingresos'], f'La leyenda debe tener una sola entrada Ingresos, tiene {entradas}'

fmt = ax1.yaxis.get_major_formatter()
assert fmt(1500000, None) == '1,500,000', \
    f'El eje Y debe mostrar miles separados: f(1500000) dio {fmt(1500000, None)!r} y se esperaba 1,500,000'
assert fmt(0, None) == '0', f'f(0) debe dar 0, dio {fmt(0, None)!r}'

assert ax1.get_ylim()[1] >= max(valores_esp) * 1.10, \
    (f'El eje Y llega hasta {ax1.get_ylim()[1]:,.0f} y la barra mas alta vale {max(valores_esp):,}: '
     'sube el limite al menos un 10% para que el texto no se salga')

print('Reto 1 resuelto.')
plt.show()

> **Si te sobra tiempo:** agrega un parametro `resaltar` con el nombre de una ciudad, que
> pinte esa barra de otro color y deje las demas en gris. Es la forma mas barata de que un
> grafico tenga un mensaje en vez de solo datos.

---
## Reto 2 - Panel de tres ejes que comparten escala

Escribe `panel_ciudades(ingresos, meses, ciudades)`.

Tres graficos de linea uno al lado del otro, uno por ciudad. El punto del reto es el eje Y
**compartido**: si cada subgrafico escala su propio eje, las tres curvas se ven igual de
altas y el panel miente. Con `sharey=True` la comparacion es honesta.

**Contrato:**
- `ingresos` es el diccionario `{ciudad: [6 valores]}`, `meses` la lista de 6 nombres,
  `ciudades` la lista de las 3 ciudades a dibujar **en ese orden**.
- Devuelve `(fig, axes)`, donde `axes` es la secuencia de los 3 `Axes` en el mismo orden.

| Requisito | Valor exacto |
|---|---|
| Disposicion | 1 fila x 3 columnas |
| Tamano | `(13, 4)` pulgadas |
| Ejes en la figura | exactamente 3 |
| Eje Y | compartido entre los tres |
| Contenido de cada eje | **una** linea, con marcador (no `'None'`) |
| Datos | los 6 ingresos de esa ciudad, en orden de mes |
| Titulo de cada eje | el nombre de la ciudad |
| Etiqueta X de cada eje | `'Mes'` |
| Etiqueta Y | `'Ingresos (COP)'` **solo en el primero**; los otros dos en `''` |
| Ticks X | los 6 nombres de mes |
| Titulo general de la figura | `'Ingresos mensuales por ciudad - primer semestre 2026'` |

La etiqueta Y solo en el primero no es pereza: con eje compartido, repetirla tres veces es
ruido. Es una decision de diseno y por eso esta en el contrato.

**Pista:** el titulo general se pone con `fig.suptitle(...)`, no con `ax.set_title(...)`.
Y si los subgraficos se pisan, `fig.tight_layout()`.

In [ ]:
def panel_ciudades(ingresos, meses, ciudades):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 2 ---
TRES = ['Tunja', 'Duitama', 'Villa de Leyva']
fig2, axes2 = panel_ciudades(INGRESOS, MESES, TRES)
fig2.canvas.draw()

ejes = list(axes2)
assert len(ejes) == 3, f'Debe devolver 3 Axes, devolvio {len(ejes)}'
assert all(isinstance(a, matplotlib.axes.Axes) for a in ejes), 'Los 3 elementos deben ser Axes'
assert len(fig2.axes) == 3, \
    f'La figura debe tener exactamente 3 ejes, tiene {len(fig2.axes)} (ojo con ejes creados de mas)'
assert tuple(round(x, 3) for x in fig2.get_size_inches()) == (13.0, 4.0), \
    f'La figura debe medir 13x4 pulgadas, no {tuple(fig2.get_size_inches())}'

titulos_fig = [t.get_text() for t in fig2.texts]
assert 'Ingresos mensuales por ciudad - primer semestre 2026' in titulos_fig, \
    f'Falta el fig.suptitle exacto del contrato. En fig.texts hay: {titulos_fig}'

for i, ciudad in enumerate(TRES):
    a = ejes[i]
    assert a.get_title() == ciudad, \
        f'El eje {i} debe titularse {ciudad!r}, dice {a.get_title()!r}'
    assert len(a.get_lines()) == 1, \
        f'El eje de {ciudad} debe tener exactamente 1 linea, tiene {len(a.get_lines())}'
    linea = a.get_lines()[0]
    ydata = [round(float(v)) for v in linea.get_ydata()]
    assert ydata == INGRESOS[ciudad], \
        f'La linea de {ciudad} lleva los datos equivocados.\n  esperado: {INGRESOS[ciudad]}\n  obtenido: {ydata}'
    assert linea.get_marker() not in ('None', None, ''), \
        f'La linea de {ciudad} debe llevar marcador (marker=...) para que se vean los 6 meses'
    assert a.get_xlabel() == 'Mes', f'El eje {i} debe llevar xlabel Mes, lleva {a.get_xlabel()!r}'
    etiquetas = [t.get_text() for t in a.get_xticklabels()]
    assert etiquetas == MESES, \
        f'Los ticks X del eje {i} deben ser los meses.\n  esperado: {MESES}\n  obtenido: {etiquetas}'

assert ejes[0].get_ylabel() == 'Ingresos (COP)', \
    f'Solo el primer eje lleva ylabel Ingresos (COP), lleva {ejes[0].get_ylabel()!r}'
assert ejes[1].get_ylabel() == '' and ejes[2].get_ylabel() == '', \
    'Los ejes 2 y 3 no llevan ylabel: con eje compartido repetirlo es ruido'

# Prueba de comportamiento: si el eje Y esta compartido, mover uno mueve a los otros.
ejes[0].set_ylim(0, 12345)
assert tuple(round(v) for v in ejes[2].get_ylim()) == (0, 12345), \
    ('Los tres ejes deben compartir el eje Y: al cambiar el limite del primero, el tercero '
     f'deberia quedar en (0, 12345) y quedo en {ejes[2].get_ylim()}. Se logra con sharey=True '
     'en plt.subplots')

print('Reto 2 resuelto.')
plt.show()

> **Si te sobra tiempo:** haz que la funcion acepte cualquier numero de ciudades y arme el
> panel en la grilla mas cuadrada posible, apagando los ejes sobrantes con `ax.set_visible(False)`.
> Ahi aparece el problema de verdad: `plt.subplots(2, 2)` devuelve un array 2D y hay que
> aplanarlo con `.ravel()`.

---
## Reto 3 - Diagrama de Pareto

Escribe `pareto_devoluciones(conteos)`.

El Pareto es el clasico que separa niveles, porque el grafico **no existe en los datos**: hay
que ordenar, acumular, normalizar a porcentaje y montar dos escalas distintas en el mismo
espacio. Dibujarlo es lo ultimo que se hace.

La regla que ilustra: unas pocas causas explican la mayoria de los casos. Las que hacen falta
para llegar al 80% acumulado son las **pocas vitales**, y son a las que se les mete plata.

**Contrato:**
- Recibe `{causa: conteo}` (usa `DEVOLUCIONES`).
- Devuelve la tupla `(fig, ax_barras, ax_pct, n_vitales)`.

| Requisito | Valor exacto |
|---|---|
| Barras | una por causa, ordenadas por conteo **descendente** |
| Ticks X | los nombres de causa en ese orden, rotados **45** grados |
| Titulo | `'Pareto de causas de devolucion'` |
| Etiqueta X / Y de las barras | `'Causa'` / `'Numero de devoluciones'` |
| `ax_pct` | un eje secundario creado con `ax_barras.twinx()` |
| Ejes en la figura | exactamente 2 |
| Lineas en `ax_pct` | exactamente 2, **en este orden**: primero la del acumulado, despues la de referencia |
| Linea 1 | porcentaje acumulado, con marcador; el ultimo punto vale `100.0` |
| Linea 2 | horizontal en el 80%, con `ax_pct.axhline(80, ...)` |
| Limites de `ax_pct` | `(0, 105)` |
| Etiqueta Y de `ax_pct` | `'Porcentaje acumulado (%)'` |
| `n_vitales` | `int`: cuantas causas hacen falta para que el acumulado **alcance o pase** el 80% |

**Ojo con `n_vitales`.** Con estos datos, tres causas suman 79.17% y cuatro suman 89.29%. La
respuesta es 4: el criterio es el primer punto que **alcanza o supera** el umbral, no el
ultimo que se queda debajo. Si te da 3, estas comparando al reves.

**Pista:** `twinx()` devuelve un `Axes` nuevo que comparte el eje X y tiene su propio eje Y a
la derecha. Dibuja las barras primero y la linea despues, si no la linea queda tapada.

In [ ]:
def pareto_devoluciones(conteos):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 3 ---
fig3, ax_b, ax_p, n_vitales = pareto_devoluciones(DEVOLUCIONES)
fig3.canvas.draw()

orden = sorted(DEVOLUCIONES.items(), key=lambda kv: -kv[1])
causas_esp = [c for c, _ in orden]
conteos_esp = [v for _, v in orden]
total = sum(conteos_esp)
acum_esp = [sum(conteos_esp[:i + 1]) / total * 100 for i in range(len(conteos_esp))]

assert isinstance(fig3, matplotlib.figure.Figure), 'El primer valor devuelto debe ser la Figure'
assert isinstance(n_vitales, int), f'n_vitales debe ser int, es {type(n_vitales).__name__}'
assert n_vitales == 4, \
    (f'n_vitales deberia ser 4 y dio {n_vitales}. Los acumulados son '
     f'{[round(p, 2) for p in acum_esp[:5]]}: la tercera causa se queda en 79.17 y no alcanza el 80')

alturas = [round(p.get_height()) for p in ax_b.patches]
assert alturas == conteos_esp, \
    f'Barras ordenadas de mayor a menor.\n  esperado: {conteos_esp}\n  obtenido: {alturas}'
etiquetas = [t.get_text() for t in ax_b.get_xticklabels()]
assert etiquetas == causas_esp, \
    f'Ticks X en el orden de las barras.\n  esperado: {causas_esp}\n  obtenido: {etiquetas}'
assert round(ax_b.get_xticklabels()[0].get_rotation()) == 45, \
    f'Los nombres de causa van rotados 45 grados, van a {ax_b.get_xticklabels()[0].get_rotation()}'
assert ax_b.get_title() == 'Pareto de causas de devolucion', \
    f'Titulo exacto del contrato. Obtenido: {ax_b.get_title()!r}'
assert ax_b.get_xlabel() == 'Causa', f'xlabel esperado Causa, obtenido {ax_b.get_xlabel()!r}'
assert ax_b.get_ylabel() == 'Numero de devoluciones', \
    f'ylabel esperado Numero de devoluciones, obtenido {ax_b.get_ylabel()!r}'

assert ax_p is not ax_b, 'ax_pct debe ser un eje distinto, creado con ax_barras.twinx()'
assert len(fig3.axes) == 2, \
    f'La figura debe quedar con exactamente 2 ejes (barras y porcentaje), tiene {len(fig3.axes)}'

lineas = ax_p.get_lines()
assert len(lineas) == 2, \
    (f'ax_pct debe tener 2 lineas (acumulado y referencia del 80%), tiene {len(lineas)}. '
     'Si tienes 1, falta el axhline; si tienes 3, dibujaste algo de mas')
acum_obt = [float(v) for v in lineas[0].get_ydata()]
assert len(acum_obt) == len(conteos_esp), \
    f'La linea del acumulado debe tener {len(conteos_esp)} puntos, tiene {len(acum_obt)}'
assert np.allclose(acum_obt, acum_esp), \
    ('La linea no lleva el porcentaje acumulado.\n  esperado: '
     f'{[round(p, 2) for p in acum_esp]}\n  obtenido: {[round(p, 2) for p in acum_obt]}')
assert abs(acum_obt[-1] - 100.0) < 1e-6, \
    f'El ultimo punto del acumulado tiene que ser 100.0, es {acum_obt[-1]}: normaliza sobre el total'
assert lineas[0].get_marker() not in ('None', None, ''), \
    'La linea del acumulado debe llevar marcador para que se vea cada causa'
assert list(np.unique(lineas[1].get_ydata())) == [80], \
    f'La segunda linea debe ser la horizontal del 80%, sus valores son {lineas[1].get_ydata()}'

assert tuple(round(v) for v in ax_p.get_ylim()) == (0, 105), \
    f'ax_pct debe ir de 0 a 105, va de {ax_p.get_ylim()}'
assert ax_p.get_ylabel() == 'Porcentaje acumulado (%)', \
    f'ylabel de ax_pct esperado Porcentaje acumulado (%), obtenido {ax_p.get_ylabel()!r}'

# Prueba de comportamiento: twinx comparte el eje X.
ax_b.set_xlim(-3, 9)
assert tuple(round(v) for v in ax_p.get_xlim()) == (-3, 9), \
    ('Los dos ejes deben compartir el eje X. Si no se movieron juntos, ax_pct no salio de '
     'ax_barras.twinx()')

print(f'Reto 3 resuelto. {n_vitales} causas explican el '
      f'{acum_esp[n_vitales - 1]:.1f}% de las devoluciones.')
plt.show()

> **Si te sobra tiempo:** pinta de un color las barras de las pocas vitales y de gris las
> demas, y mete el umbral como parametro `umbral=80`. Un Pareto al 95% cuenta otra historia y
> el codigo no deberia cambiar.

---
## Reto 4 - Un tema propio, reutilizable

Escribe `aplicar_estilo(ax, formato_miles=True)`. Es el reto mas duro de los cuatro y el que
mas va a pesar en la fase de visualizacion del Proyecto (Actividad 3), donde el 25% de la nota es *personalizacion avanzada*.

La tentacion es `plt.style.use('seaborn-v0_8')` o `sns.set_theme()`. El problema es que ambas
son **globales**: cambian el proceso entero y no se pueden aplicar a un solo grafico ni
componer con otras. Lo que se hace en produccion es una funcion que recibe un `Axes` y lo
deja con la identidad visual de la casa.

**Contrato:**
- Recibe un `Axes` **ya dibujado** y lo modifica en el sitio.
- **Devuelve el mismo `ax`** que recibio, para poder encadenar.
- No toca los datos: no agrega ni quita lineas, barras ni textos.
- No toca ningun otro eje de la figura ni el estado global de matplotlib.

| Requisito | Verificacion |
|---|---|
| Marco superior y derecho ocultos | `ax.spines['top'].get_visible()` da `False` |
| Marco izquierdo e inferior visibles, color `'#444444'`, grosor `0.8` | `ax.spines['left'].get_edgecolor()` |
| Rejilla **solo horizontal** | `ax.get_ygridlines()[0].get_visible()` da `True`, la de X `False` |
| Rejilla suave | `alpha` entre `0.2` y `0.5`, grosor `<= 0.8` |
| Rejilla detras de los datos | `ax.get_axisbelow()` da `True` |
| Titulo a 13 puntos y en negrita, **sin cambiar el texto** | `ax.title.get_fontsize()`, `.get_fontweight()` |
| Etiquetas de tick a 10 puntos | `ax.get_xticklabels()[0].get_fontsize()` |
| Marcas del eje Y eliminadas, las de X intactas | `ax.yaxis.get_major_ticks()[0].tick1line.get_visible()` |
| Con `formato_miles=True`, eje Y con separador de miles | `f(1500000, None) == '1,500,000'` |
| Con `formato_miles=False`, el formateador **no se toca** | debe seguir siendo el mismo objeto |
| Idempotente | aplicarla dos veces da el mismo resultado |

**Pistas:**

- Las marcas de tick se apagan con `ax.tick_params(...)`. Busca que parametro apaga solo las
  del lado izquierdo. Ojo: `length=0` las deja invisibles a la vista pero el objeto sigue
  reportandose como visible, y el assert mira el objeto.
- `ax.grid(...)` acumula. Si el `Axes` que te llega ya traia rejilla vertical, ponerle rejilla
  horizontal no le quita la otra. Apaga todo primero.
- El titulo se restila sin reescribirlo: `ax.title` es el objeto `Text`, y tiene
  `set_fontsize` y `set_fontweight`.

El ultimo assert es el que de verdad prueba que la funcion es local: aplica el estilo a **un**
eje de una figura de dos y comprueba que el otro quedo intacto. Si usaste `plt.style.use` o
`sns.set_theme`, ahi se cae.

In [ ]:
def aplicar_estilo(ax, formato_miles=True):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 4 ---
import matplotlib.colors as mcolors

fig4, (ax_izq, ax_der) = plt.subplots(1, 2, figsize=(11, 4))
ax_izq.plot(MESES, INGRESOS['Tunja'], marker='o')
ax_izq.set_title('Tunja mes a mes')
ax_izq.grid(True)                      # rejilla en los dos ejes: el estilo debe dejar solo la Y
ax_der.plot(MESES, INGRESOS['Duitama'], marker='s')

n_lineas_antes = len(ax_izq.get_lines())
devuelto = aplicar_estilo(ax_izq)
fig4.canvas.draw()

assert devuelto is ax_izq, 'La funcion debe devolver el mismo Axes que recibio, para encadenar'
assert len(ax_izq.get_lines()) == n_lineas_antes, \
    'La funcion de estilo no debe agregar ni quitar datos del eje'
assert ax_izq.get_title() == 'Tunja mes a mes', \
    f'El texto del titulo no se toca, solo su formato. Quedo en {ax_izq.get_title()!r}'

assert ax_izq.spines['top'].get_visible() is False, 'El marco superior debe quedar oculto'
assert ax_izq.spines['right'].get_visible() is False, 'El marco derecho debe quedar oculto'
assert ax_izq.spines['left'].get_visible() is True, 'El marco izquierdo se conserva'
assert ax_izq.spines['bottom'].get_visible() is True, 'El marco inferior se conserva'
gris = mcolors.to_rgba('#444444')
for lado in ('left', 'bottom'):
    color = mcolors.to_rgba(ax_izq.spines[lado].get_edgecolor())
    assert color == gris, f'El marco {lado} debe ser #444444, es {color}'
    assert abs(ax_izq.spines[lado].get_linewidth() - 0.8) < 1e-9, \
        f'El marco {lado} debe tener grosor 0.8, tiene {ax_izq.spines[lado].get_linewidth()}'

gy = ax_izq.get_ygridlines()
gx = ax_izq.get_xgridlines()
assert gy and gy[0].get_visible(), 'Debe quedar rejilla horizontal (axis=y)'
assert not gx[0].get_visible(), \
    'No debe quedar rejilla vertical. El eje ya traia grid(True): apagala antes de poner la tuya'
alpha = gy[0].get_alpha()
assert alpha is not None and 0.2 <= alpha <= 0.5, \
    f'La rejilla debe ser suave: alpha entre 0.2 y 0.5, tiene {alpha}'
assert gy[0].get_linewidth() <= 0.8, \
    f'La rejilla debe ser delgada: grosor <= 0.8, tiene {gy[0].get_linewidth()}'
assert ax_izq.get_axisbelow() is True, \
    'La rejilla debe quedar detras de los datos: ax.set_axisbelow(True)'

assert ax_izq.title.get_fontsize() == 13, \
    f'El titulo va a 13 puntos, esta a {ax_izq.title.get_fontsize()}'
assert ax_izq.title.get_fontweight() in ('bold', 700), \
    f'El titulo va en negrita, esta en {ax_izq.title.get_fontweight()!r}'
assert ax_izq.get_xticklabels()[0].get_fontsize() == 10, \
    f'Las etiquetas de tick van a 10 puntos, estan a {ax_izq.get_xticklabels()[0].get_fontsize()}'

assert ax_izq.yaxis.get_major_ticks()[0].tick1line.get_visible() is False, \
    'Las marcas del eje Y deben desaparecer. Revisa que parametro de ax.tick_params las apaga'
assert ax_izq.xaxis.get_major_ticks()[0].tick1line.get_visible() is True, \
    'Las marcas del eje X se conservan: solo se quitan las del Y'

f4 = ax_izq.yaxis.get_major_formatter()
assert f4(1500000, None) == '1,500,000', \
    f'Con formato_miles=True el eje Y separa miles: f(1500000) dio {f4(1500000, None)!r}'

# El otro eje de la MISMA figura tiene que haber quedado intacto.
assert ax_der.spines['top'].get_visible() is True, \
    ('El estilo se le aplico al eje de al lado. Estas usando algo global (plt.style.use, '
     'sns.set_theme o plt.rcParams) en vez de tocar solo el ax que recibes')
assert not ax_der.get_ygridlines()[0].get_visible(), \
    'El eje de al lado no debe quedar con rejilla: la funcion actua solo sobre el ax recibido'

# formato_miles=False no toca el formateador.
fig5, ax5 = plt.subplots()
ax5.plot([1, 2, 3], [10, 20, 30])
fmt_antes = ax5.yaxis.get_major_formatter()
aplicar_estilo(ax5, formato_miles=False)
assert ax5.yaxis.get_major_formatter() is fmt_antes, \
    'Con formato_miles=False el formateador del eje Y se queda como estaba'
assert ax5.spines['top'].get_visible() is False, \
    'formato_miles=False solo afecta al formateador: el resto del estilo se aplica igual'

# Idempotente: dos pasadas dejan lo mismo que una.
aplicar_estilo(ax_izq)
assert ax_izq.spines['top'].get_visible() is False and ax_izq.get_axisbelow() is True, \
    'Aplicar el estilo dos veces debe dar el mismo resultado que aplicarlo una'

# Y tiene que servir sobre un Axes dibujado por seaborn, que tambien es un Axes normal.
fig6, ax6 = plt.subplots(figsize=(7, 4))
sns.barplot(data=ventas, x='ciudad', y='ingreso', errorbar=None, ax=ax6)
ax6.set_title('Ingreso mensual promedio por ciudad')
aplicar_estilo(ax6)
assert ax6.spines['right'].get_visible() is False, \
    'La funcion debe servir sobre cualquier Axes, lo haya dibujado matplotlib o seaborn'
assert len(ax6.patches) == 5, 'El estilo no debe borrar las barras que dibujo seaborn'

print('Reto 4 resuelto.')
plt.show()

> **Si te sobra tiempo:** saca los valores fijos a un diccionario `TEMA` con la paleta, los
> tamanos y los colores, y que `aplicar_estilo` lo reciba como parametro con un valor por
> defecto. Ahi tienes un tema corporativo de verdad: cambias el diccionario y cambian todos
> los graficos del informe.

---
## Reto abierto - sin verificacion automatica

Los cuatro retos anteriores tenian una respuesta correcta. Este no.

**El encargo.** La gerencia de la cadena pregunta:

> *Villa de Leyva crecio 84% en el semestre y Tunja apenas 11%. Deberiamos mover inversion
> de Tunja a Villa de Leyva?*

Tienes la tabla `ventas` (30 filas: ciudad, mes, ingreso, unidades, devoluciones,
satisfaccion). Construye **un solo grafico** que le sirva a esa persona para decidir, y
escribe debajo el parrafo de interpretacion.

**Requisitos minimos:**

- Una sola figura. Puede tener varios ejes si lo justificas, pero un solo mensaje.
- Al menos un elemento de seaborn.
- Pasala por tu `aplicar_estilo`.
- Que se lea sin que tu la expliques: titulos, unidades, y lo importante resaltado.

**Ideas, no todas buenas:**

- Crecimiento porcentual contra volumen absoluto: 84% sobre una base chica puede ser menos
  plata que 11% sobre una base grande. Ese es probablemente el nudo del asunto.
- Ingreso por unidad vendida (el ticket real) por ciudad.
- Tasa de devolucion contra satisfaccion: crecer con el servicio roto no es crecer.
- Las seis curvas mensuales normalizadas a base 100 en enero.

Se juzga el **criterio**: que decidiste mostrar, que decidiste dejar por fuera, y si el
grafico responde la pregunta que hicieron o una distinta.

In [ ]:
# tu codigo aqui
# Construye la figura, aplicale tu estilo y muestrala.




**Tu interpretacion** (esto es exactamente lo que pide el 25% de *interpretacion critica*
de la fase de visualizacion del Proyecto (Actividad 3); escribe entre 5 y 8 lineas):

- Que muestra el grafico:
- Que decision respalda:
- Que **no** se puede concluir con estos datos:

**Tus decisiones de diseno:**

1. Elegi este tipo de grafico porque
2. Deje por fuera
3. Resalte

---
## Como se conecta con tu Proyecto (Actividad 3)

La visualizacion es una **fase del Proyecto (Actividad 3, 40%)**; se ensaya aqui, sin entrega aparte, y su
rubrica son cuatro criterios de 25% cada uno:

| Criterio de la rubrica | Que hiciste hoy | Que te falta |
|---|---|---|
| Personalizacion avanzada | Retos 1 y 4: contrato exacto de textos, ticks, formato y tema reutilizable | Aplicarlo a todos los graficos de la entrega, no a uno |
| Graficos variados | Barras, lineas, panel, Pareto, mapa de correlacion | Que cada uno responda una pregunta distinta y se note |
| Interactividad con Plotly | nada | **Sesion 6** |
| Interpretacion critica | El parrafo del reto abierto | Un parrafo por grafico, no uno al final |

Dos advertencias que cuestan puntos:

1. **Las tres librerias son obligatorias por rubrica: matplotlib, seaborn Y plotly.** Con
   matplotlib y seaborn impecables te quedas sin un cuarto de la nota. Plotly llega la
   proxima sesion; no lo dejes para la vispera.
2. La interpretacion no es describir el grafico. "Las barras de Tunja son mas altas" no es
   interpretacion, es leer en voz alta. Interpretacion es decir que se hace con eso y que
   riesgo tiene.

Lo que te llevas de hoy que no estaba en el temario: tus graficos devuelven `fig, ax`. Eso
significa que en la entrega puedes tener una funcion por grafico, un `aplicar_estilo` comun y
un notebook que se relee. Es la diferencia entre entregar graficos y entregar un informe.

---
## Para terminar

1. Ejecuta el notebook completo de arriba abajo y confirma que las cuatro verificaciones pasan.
2. Guardalo **con las salidas visibles** (Archivo > Guardar). Si las figuras no se ven, el
   revisor no ve nada: revisa que cada celda de verificacion termine mostrando su grafico.
3. Si quieres, subelo al aula (opcional, no se califica) y lo reviso y te comento por escrito.

Si terminaste temprano: coge `aplicar_estilo` y aplicaselo a los graficos del reto 1, 2 y 3.
Vas a encontrar que en el panel de tres ejes hay que decidir si el estilo va por eje o por
figura, y que en el Pareto hay dos ejes Y y uno de ellos no quiere rejilla. Esas dos
decisiones son el 25% de personalizacion avanzada de la actividad.

**Proxima sesion:** Plotly e interactividad. La via avanzada arranca donde esta termina.

*Dudas: consulta al profesor*